# OCR Image — Donut backend on Google Colab (free)

Runs the original **Donut** receipt model as an API and exposes a public URL
via a free Cloudflare quick tunnel (no account or token needed).

**How to use:**
1. Runtime → *Run all* (or run each cell top to bottom).
2. Wait for the last cell to print a `https://....trycloudflare.com` URL.
3. Put that URL in `BACKEND_URL` at the top of `index.html` in your repo
   (or just open the URL directly — it serves the web UI too).

The URL stays live only while this notebook is running. It changes each time
you restart, so paste the fresh one when you re-run. For speed, use
*Runtime → Change runtime type → T4 GPU* (still free).

In [ ]:
# 1. Install dependencies
!pip -q install flask flask-cors transformers Pillow sentencepiece protobuf

In [ ]:
# 2. Fetch the app code from the public repo (main branch)
!wget -q https://raw.githubusercontent.com/mmedabo/ocr-image/main/ocr_receipt.py -O ocr_receipt.py
!wget -q https://raw.githubusercontent.com/mmedabo/ocr-image/main/app.py -O app.py
!mkdir -p templates
!wget -q https://raw.githubusercontent.com/mmedabo/ocr-image/main/templates/index.html -O templates/index.html
print('App code downloaded.')

In [ ]:
# 3. Download cloudflared (for the free public tunnel)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
print('cloudflared ready.')

In [ ]:
# 4. Load the Donut model, start the API, and open the public tunnel
import subprocess, threading, time, re
import ocr_receipt

print('Loading Donut model (one-time, ~1 GB download)...')
ocr_receipt._load_model()
print('Model loaded.')

from app import app
threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=5000, use_reloader=False),
    daemon=True,
).start()
time.sleep(3)
print('Flask API running on :5000')

# Start a Cloudflare quick tunnel and capture the public URL from its log.
logf = open('cf.log', 'w')
subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:5000', '--no-autoupdate'],
    stdout=logf, stderr=logf,
)

url = None
for _ in range(40):
    time.sleep(1)
    try:
        m = re.search(r'https://[-\w]+\.trycloudflare\.com', open('cf.log').read())
        if m:
            url = m.group(0)
            break
    except FileNotFoundError:
        pass

print('\n' + '=' * 60)
if url:
    print('PUBLIC URL:', url)
    print('\nOpen it directly for the web UI, or set it as BACKEND_URL')
    print('at the top of index.html to power your GitHub Pages app.')
else:
    print('Tunnel URL not found yet. Re-run this cell, or check cf.log.')
print('=' * 60)
print('\nKeep this notebook running to keep the URL alive.')

## Test it

- Open the printed URL in a browser — you get the same drag-and-drop UI,
  now backed by Donut.
- Or point your GitHub Pages app at it: edit `index.html`,
  `const BACKEND_URL = "https://....trycloudflare.com";`, commit, and the
  hosted app will send images here for Donut parsing (falling back to
  in-browser Tesseract if this notebook is off).